In [1]:
# ==================================================
# Project Path
# ==================================================

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]

sys.path.insert(
    0,
    str(PROJECT_ROOT)
)

In [2]:
# ==================================================
# Import Libraries
# ==================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from xgboost import XGBClassifier

import mlflow

from src.mlflow.tracking import MLflowTracker
from src.mlflow.utils import classification_metrics

In [3]:
# ==================================================
# Load Dataset
# ==================================================

DATA_PATH = (
    r"D:\Subject\CV2026\Market Risk Classification"
    r"\market-risk-classification"
    r"\data\processed\BTCUSDT\BTCUSDT_1m_clean.csv"
)

print("="*60)
print("Loading Dataset ...")
print("="*60)

df = pd.read_csv(DATA_PATH)

print(df.shape)

df.head()

Loading Dataset ...
(2000000, 12)


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,2017-08-17 04:00:00,4261.48,4261.48,4261.48,4261.48,1.775183,2017-08-17 04:00:59.999,7564.906851,3,0.075183,320.390851,0
1,2017-08-17 04:01:00,4261.48,4261.48,4261.48,4261.48,0.000000,2017-08-17 04:01:59.999,0.000000,0,0.000000,0.000000,0
2,2017-08-17 04:02:00,4280.56,4280.56,4280.56,4280.56,0.261074,2017-08-17 04:02:59.999,1117.542921,2,0.261074,1117.542921,0
3,2017-08-17 04:03:00,4261.48,4261.48,4261.48,4261.48,0.012008,2017-08-17 04:03:59.999,51.171852,3,0.012008,51.171852,0
4,2017-08-17 04:04:00,4261.48,4261.48,4261.48,4261.48,0.140796,2017-08-17 04:04:59.999,599.999338,1,0.140796,599.999338,0


In [4]:
# ==================================================
# Datetime
# ==================================================

df["open_time"] = pd.to_datetime(df["open_time"])

df = df.sort_values(
    "open_time"
).reset_index(drop=True)

df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,2017-08-17 04:00:00,4261.48,4261.48,4261.48,4261.48,1.775183,2017-08-17 04:00:59.999,7564.906851,3,0.075183,320.390851,0
1,2017-08-17 04:01:00,4261.48,4261.48,4261.48,4261.48,0.000000,2017-08-17 04:01:59.999,0.000000,0,0.000000,0.000000,0
2,2017-08-17 04:02:00,4280.56,4280.56,4280.56,4280.56,0.261074,2017-08-17 04:02:59.999,1117.542921,2,0.261074,1117.542921,0
3,2017-08-17 04:03:00,4261.48,4261.48,4261.48,4261.48,0.012008,2017-08-17 04:03:59.999,51.171852,3,0.012008,51.171852,0
4,2017-08-17 04:04:00,4261.48,4261.48,4261.48,4261.48,0.140796,2017-08-17 04:04:59.999,599.999338,1,0.140796,599.999338,0


In [5]:
# ==================================================
# Drop Columns
# ==================================================

df = df.drop(
    columns=[
        "close_time",
        "ignore"
    ]
)

print(df.shape)

df.head()

(2000000, 10)


,open_time,open,high,low,close,volume,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
0,2017-08-17 04:00:00,4261.48,4261.48,4261.48,4261.48,1.775183,7564.906851,3,0.075183,320.390851
1,2017-08-17 04:01:00,4261.48,4261.48,4261.48,4261.48,0.000000,0.000000,0,0.000000,0.000000
2,2017-08-17 04:02:00,4280.56,4280.56,4280.56,4280.56,0.261074,1117.542921,2,0.261074,1117.542921
3,2017-08-17 04:03:00,4261.48,4261.48,4261.48,4261.48,0.012008,51.171852,3,0.012008,51.171852
4,2017-08-17 04:04:00,4261.48,4261.48,4261.48,4261.48,0.140796,599.999338,1,0.140796,599.999338


In [6]:
# ==================================================
# Create Targets
# ==================================================

HORIZONS = [
    1,
    5,
    10,
    15,
    30
]

print("="*60)
print("Creating Targets...")
print("="*60)

for h in tqdm(HORIZONS):

    df[f"target_{h}m"] = (

        df["close"].shift(-h)

        >

        df["close"]

    ).astype(int)

print(df.shape)

df.head()

Creating Targets...


100%|██████████| 5/5 [00:00<00:00, 38.26it/s]

(2000000, 15)


,open_time,open,high,low,close,volume,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,target_1m,target_5m,target_10m,target_15m,target_30m
0,2017-08-17 04:00:00,4261.48,4261.48,4261.48,4261.48,1.775183,7564.906851,3,0.075183,320.390851,0,0,0,0,1
1,2017-08-17 04:01:00,4261.48,4261.48,4261.48,4261.48,0.000000,0.000000,0,0.000000,0.000000,1,0,0,0,1
2,2017-08-17 04:02:00,4280.56,4280.56,4280.56,4280.56,0.261074,1117.542921,2,0.261074,1117.542921,0,0,0,0,0
3,2017-08-17 04:03:00,4261.48,4261.48,4261.48,4261.48,0.012008,51.171852,3,0.012008,51.171852,0,0,0,1,1
4,2017-08-17 04:04:00,4261.48,4261.48,4261.48,4261.48,0.140796,599.999338,1,0.140796,599.999338,0,0,0,0,1


In [7]:
# ==================================================
# Remove Tail NaN
# ==================================================

df = df.iloc[:-max(HORIZONS)]

print(df.shape)

(1999970, 15)


In [8]:
# ==================================================
# Features
# ==================================================

FEATURES = [

    "open",

    "high",

    "low",

    "close",

    "volume",

    "quote_asset_volume",

    "number_of_trades",

    "taker_buy_base_volume",

    "taker_buy_quote_volume",

]

FEATURES

['open',
 'high',
 'low',
 'close',
 'volume',
 'quote_asset_volume',
 'number_of_trades',
 'taker_buy_base_volume',
 'taker_buy_quote_volume']

In [9]:
# ==================================================
# Target Distribution
# ==================================================

for h in HORIZONS:

    print("="*50)

    print(f"Target {h} minutes")

    print(
        df[
            f"target_{h}m"
        ].value_counts()
    )

Target 1 minutes
target_1m
0    1028393
1     971577
Name: count, dtype: int64
Target 5 minutes
target_5m
1    1000729
0     999241
Name: count, dtype: int64
Target 10 minutes
target_10m
1    1008621
0     991349
Name: count, dtype: int64
Target 15 minutes
target_15m
1    1013436
0     986534
Name: count, dtype: int64
Target 30 minutes
target_30m
1    1019660
0     980310
Name: count, dtype: int64


In [10]:
# ==================================================
# Initialize MLflow
# ==================================================

tracker = MLflowTracker()

In [11]:
# ==================================================
# XGBoost Parameters
# ==================================================

XGB_PARAMS = {

    "n_estimators": 300,

    "max_depth": 6,

    "learning_rate": 0.05,

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "objective": "binary:logistic",

    "eval_metric": "logloss",

    "random_state": 42,

    "tree_method": "hist",

    "n_jobs": -1

}

In [12]:
# ==================================================
# Training Summary
# ==================================================

summary = []

In [13]:
# ==================================================
# Train XGBoost
# ==================================================

for horizon in tqdm(
    HORIZONS,
    desc="Training Models"
):

    print("="*60)
    print(f"Training Horizon : {horizon} minutes")
    print("="*60)

    TARGET = f"target_{horizon}m"

    X = df[FEATURES]

    y = df[TARGET]

    # ----------------------------------------
    # Train Test Split
    # ----------------------------------------

    X_train, X_test, y_train, y_test = train_test_split(

        X,

        y,

        test_size=0.20,

        shuffle=False

    )

    # ----------------------------------------
    # Model
    # ----------------------------------------

    model = XGBClassifier(
        **XGB_PARAMS
    )

    # ----------------------------------------
    # MLflow
    # ----------------------------------------

    with tracker.start_run(

        run_name=f"XGBoost_Return_{horizon}m"

    ):

        model.fit(

            X_train,

            y_train,

            verbose=False

        )

        y_pred = model.predict(

            X_test

        )

        y_prob = model.predict_proba(

            X_test

        )[:,1]

        metrics = classification_metrics(

            y_test,

            y_pred,

            y_prob

        )

        print(metrics)
        print(metrics.keys())

        # ----------------------------
        # Log Parameters
        # ----------------------------

        tracker.log_params({

            **XGB_PARAMS,

            "prediction_horizon": horizon,

            "feature_count": len(FEATURES)

        })

        # ----------------------------
        # Log Metrics
        # ----------------------------

        tracker.log_metrics(

            metrics

        )

        # ----------------------------
        # Save Model
        # ----------------------------

        tracker.log_model(

            model

        )

        summary.append({

            "Model": f"Return_{horizon}m",

            "Accuracy": metrics["accuracy"],

            "Precision": metrics["precision"],

            "Recall": metrics["recall"],

            "F1": metrics["f1"],

            "ROC-AUC": metrics["roc_auc"]

        })

Training Models:   0%|          | 0/5 [00:00<?, ?it/s]

Training Horizon : 1 minutes
{'accuracy': 0.49876748151222267, 'precision': 0.4972984001995028, 'recall': 0.6611775344117351, 'f1': 0.5676467163232827, 'roc_auc': 0.4992749051320467}
dict_keys(['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])


Training Models:  20%|██        | 1/5 [00:45<03:00, 45.21s/it]

Training Horizon : 5 minutes
{'accuracy': 0.49989499842497637, 'precision': 0.5098174599749709, 'recall': 0.11758207306331457, 'f1': 0.19109161123354698, 'roc_auc': 0.5037596882253128}
dict_keys(['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])


Training Models:  40%|████      | 2/5 [01:02<01:26, 28.75s/it]

...

Training Models:  60%|██████    | 3/5 [01:20<00:47, 23.86s/it]

Training Horizon : 15 minutes
{'accuracy': 0.4967249508742631, 'precision': 0.5141535962007123, 'recall': 0.12179645327780683, 'f1': 0.19694026129450484, 'roc_auc': 0.5052718092669642}
dict_keys(['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])


Training Models:  80%|████████  | 4/5 [01:41<00:22, 22.73s/it]

Training Horizon : 30 minutes
{'accuracy': 0.4938124071861078, 'precision': 0.5212590130587188, 'recall': 0.14012782391742712, 'f1': 0.22087797069326437, 'roc_auc': 0.5032187453145184}
dict_keys(['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])


Training Models: 100%|██████████| 5/5 [02:03<00:00, 24.71s/it]


In [14]:
# ==================================================
# Summary
# ==================================================

summary = pd.DataFrame(summary)

summary = summary.sort_values(

    by="Accuracy",

    ascending=False

)

summary.reset_index(

    drop=True,

    inplace=True

)

summary

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Return_10m,0.500885,0.503873,0.623181,0.557212,0.500598
1,Return_5m,0.499895,0.509817,0.117582,0.191092,0.503760
2,Return_1m,0.498767,0.497298,0.661178,0.567647,0.499275
3,Return_15m,0.496725,0.514154,0.121796,0.196940,0.505272
4,Return_30m,0.493812,0.521259,0.140128,0.220878,0.503219


In [15]:
# ==================================================
# Generate Summary
# ==================================================

tracker.generate_summary()

MLflow training summary generated
Saved at: D:\Subject\CV2026\Market Risk Classification\market-risk-classification\artifacts\training_summary.txt


In [16]:
# ==================================================
# MLflow Runs
# ==================================================

experiment = mlflow.get_experiment_by_name(

    "Market Risk Classification"

)

runs = mlflow.search_runs(

    experiment_ids=[

        experiment.experiment_id

    ]

)

runs[
[
    "run_id",

    "tags.mlflow.runName",

    "metrics.accuracy",

    "metrics.precision",

    "metrics.recall",

    "metrics.f1",

    "metrics.roc_auc"

]]

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1,metrics.roc_auc
0,81b0cb8ed06141149b5a4fde13d04377,XGBoost_Return_30m,0.493812,0.521259,0.140128,0.220878,0.503219
1,773c2125e2924bc6a9c974eb450d9d41,XGBoost_Return_15m,0.496725,0.514154,0.121796,0.196940,0.505272
2,2c8d076419594c81adc3ab3fb87023a8,XGBoost_Return_10m,0.500885,0.503873,0.623181,0.557212,0.500598
3,43e3443f1b08484b92fb6f2b82d96206,XGBoost_Return_5m,0.499895,0.509817,0.117582,0.191092,0.503760
4,f09402f135bf49569fc47b8a44a0ac8a,XGBoost_Return_1m,0.498767,0.497298,0.661178,0.567647,0.499275
5,76ebd23913a0421a82f8294da23f692f,XGBoost_Return_1m,0.498767,0.497298,0.661178,NaN,0.499275
6,cd33862de4bb4a9caa75a5b761a9f29a,XGBoost_Return_1m,0.498767,0.497298,0.661178,NaN,0.499275
7,dd20f85fe6da4466ae9207f0d140cbf9,XGBoost_Return_1m,0.498767,0.497298,0.661178,NaN,0.499275
8,5527a31df31a499582e883b422d24e42,Logistic Regression Feature Store,0.497459,0.496933,0.782137,NaN,0.494186
9,b001b6536bce4e50b61f667b72d53118,Logistic Regression Feature Store,0.497459,0.496933,0.782137,NaN,0.494186


In [17]:
# ==================================================
# Best Horizon
# ==================================================

summary.iloc[0]

Model        Return_10m
Accuracy       0.500885
Precision      0.503873
Recall         0.623181
F1             0.557212
ROC-AUC        0.500598
Name: 0, dtype: object